# Tucker rank / 再構成誤差の確認

このNotebookでは、`00_tucker_hosvd_basics.ipynb` で作ったHOSVD/Tuckerの自作関数を使って、**multilinear rankを変えると「保存するパラメータ数」と「再構成誤差」がどう変わるか**を確認する。

このNotebookで新しい分解アルゴリズムは作らない。やることは次の4つだけ。

1. 同じテンソルを複数のrank設定でHOSVDする
2. `core + factor matrices` の総要素数を数える
3. 元テンソルを再構成して relative Frobenius error を求める
4. rankを小さくしたとき、サイズと誤差がどう変化するか比較する

最終的には、この考え方をCNNのTucker-2で使う `out channel rank` / `in channel rank` の選択につなげる。

> **このNotebookのゴール**  
> 「rankをどこまで下げれば小さくできるか」と「その代わりにどれだけ近似誤差が増えるか」を、数値で説明できるようになること。


## 1. 実験対象テンソル

rankの違いだけを比較したいので、固定seedで3階テンソル `X` を1つ作る。

`X.shape = (6, 5, 4)` なので、元テンソルが持つ要素数は

\[
6 \times 5 \times 4 = 120
\]

となる。

この `120` を基準に、Tucker表現が何要素まで小さくなるかを比較する。


In [ ]:
import torch

torch.manual_seed(0)
X = torch.randn(6, 5, 4)

print("shape:", X.shape)
print("elements:", X.numel())


## 2. `00_tucker_hosvd_basics.ipynb` の自作関数を使えるようにする

このNotebookは別Notebookなので、`00_tucker_hosvd_basics.ipynb` で定義した関数は自動では引き継がれない。

下のコードセルには、**前Notebookで完成した実装をそのまま持ってくる**。

今回必要なのは次の処理。

- `truncated_svd`
- `unfold`
- `fold`
- `mode_dot`
- `hosvd`
- `reconstruct_tucker`
- `relative_frobenius_error`

`hosvd` や `reconstruct_tucker` だけをコピーすると、その内部で使う `unfold` / `mode_dot` などが存在せず動かないので注意する。

> ここでは新しく書き直さない。`00_tucker_hosvd_basics.ipynb` で完成したものを再利用できればよい。  
> 後で `src` に共通化したら、このセルはimportへ置き換える。


## 3. 比較するrankを決める

まずは **全modeを圧縮するHOSVD** を比較する。

元のshapeは `(6, 5, 4)` なので、完全rankは

```python
{0: 6, 1: 5, 2: 4}
```

になる。

そこから少しずつrankを下げる。

各辞書は、

```text
mode 0 のrank
mode 1 のrank
mode 2 のrank
```

を表している。

この段階では結果を予想しなくてよい。後で各設定について同じ処理を行い、パラメータ数と誤差を比較する。


In [ ]:
rank_settings = [
    {0: 6, 1: 5, 2: 4},
    {0: 4, 1: 3, 2: 3},
    {0: 3, 1: 2, 2: 2},
    {0: 2, 1: 2, 2: 1},
]

rank_settings


## 4. Tucker表現のパラメータ数を数える

ここでは `tucker_parameter_count(shape, ranks)` を完成させる。

Tucker分解後に保存するのは、

1. **core tensor**
2. **分解した各modeのfactor matrix**

の2種類。

### 4.1 core tensorの要素数

元shapeが

\[
(I_0, I_1, \dots)
\]

で、あるmodeをrank \(r_n\) に圧縮した場合、そのmodeのcoreサイズは \(I_n\) から \(r_n\) に変わる。

例えば

```python
shape = (6, 5, 4)
ranks = {0: 3, 1: 2, 2: 2}
```

なら、

```text
core.shape = (3, 2, 2)
```

なのでcoreの要素数は

\[
3 \times 2 \times 2
\]

となる。

partial HOSVDのように `ranks` に含まれていないmodeは、元のdimensionをそのままcoreに残す。

### 4.2 factor matrixの要素数

mode \(n\) の元サイズが \(I_n\)、指定rankが \(r_n\) ならfactor matrixは

\[
U_n \in \mathbb{R}^{I_n \times r_n}
\]

なので、要素数は

\[
I_n r_n
\]

となる。

したがって、この関数では

\[
\text{Tucker params}
=
\text{coreの要素数}
+
\sum_{n \in \text{圧縮mode}} I_n r_n
\]

を返せばよい。

**この式をそのままコードにする**のがこの節の作業。


In [ ]:
def tucker_parameter_count(shape, ranks):
    """
    目的: 指定したrankでTucker表現を作ったときの、
          core tensorとfactor matrixの総要素数を求める。

    shape: 元テンソルのshape
    ranks: {mode: rank} 形式の辞書
    """
    pass


## 5. 各rankで「サイズ」と「誤差」を計算する

ここがこのNotebookの中心。

`rank_settings` の各rankについて、**必ず同じ順番で**次を計算する。

1. `hosvd(X, ranks)` で `core, factors` を得る
2. `reconstruct_tucker(core, factors)` で `X_hat` を作る
3. `relative_frobenius_error(X, X_hat)` で再構成誤差を求める
4. `tucker_parameter_count(X.shape, ranks)` でTucker表現の要素数を求める
5. 元テンソルの要素数 `X.numel()` と比較する
6. rank・core shape・要素数・誤差を1つの結果として保存する

比較するときに曖昧にならないよう、このNotebookでは次の2つを区別する。

### parameter ratio

\[
\frac{\text{Tucker params}}{\text{original params}}
\]

元の何割の要素数になったかを表す。**小さいほど圧縮できている**。

### compression factor

\[
\frac{\text{original params}}{\text{Tucker params}}
\]

元に比べて何倍小さい表現になったかを表す。**大きいほど圧縮できている**。

最低限、次の項目がrankごとに分かるように結果を保存する。

```text
ranks
core_shape
tucker_params
parameter_ratio
compression_factor
relative_error
```

結果の保存方法は `list[dict]` でもよいし、その場で `print` してもよい。重要なのは、**全rank設定を同じ指標で比較できること**。


## 6. full HOSVDの結果を読む

Section 5 の結果を見て、rankを下げたときの変化を確認する。

見るものは次の4つ。

- `core_shape` は小さくなっているか
- `tucker_params` / `parameter_ratio` は小さくなっているか
- `compression_factor` は大きくなっているか
- `relative_error` はどのように変わったか

ここで重要なのは、単に「一番小さいrankが良い」と決めることではない。

例えば、

```text
rankを少し下げる
→ パラメータ数は大きく減る
→ 誤差はほとんど増えない
```

なら、そのrankは有力。

逆に、

```text
rankをさらに下げる
→ パラメータ数は少ししか減らない
→ 誤差が急に増える
```

なら、下げすぎている可能性がある。

つまりこの節では、**圧縮量と再構成誤差のtrade-offを見る**。

空のコードセルには、Section 5 で保存した結果を見やすく表示する処理を書く。表にしても、1行ずつprintしてもよい。


## 7. partial HOSVDでも同じ比較をする

次は **mode 0 と mode 1 だけを圧縮し、mode 2 は圧縮しない**。

これはCNNのTucker-2への準備。

Conv2dの重みは最終的に

\[
(C_{out}, C_{in}, k_H, k_W)
\]

という4階テンソルとして扱う。

Tucker-2では主に、

- mode 0: `out channel`
- mode 1: `in channel`

だけを低rank化し、kernel方向の `kH`, `kW` はそのまま残す。

この3階テンソルでは、その簡略版としてmode 0, 1だけを圧縮する。

例えば、

```python
ranks = {0: 3, 1: 2}
```

ならmode 2は指定されていないので、

```text
元:   (6, 5, 4)
core: (3, 2, 4)
```

となる。

下の `partial_rank_settings` について、Section 5 と同じことを行う。

1. partial HOSVD
2. 再構成
3. Tuckerパラメータ数
4. parameter ratio / compression factor
5. relative Frobenius error

そして、全modeを圧縮した場合との違いを見る。


In [ ]:
partial_rank_settings = [
    {0: 6, 1: 5},
    {0: 4, 1: 3},
    {0: 3, 1: 2},
    {0: 2, 1: 2},
]

partial_rank_settings


## 8. CNNへ進む前の確認

このNotebookでは、関数を新しく増やすことより、**rankを変えた結果を読めること**が重要。

最後に、自分の実験結果を見ながら次を説明する。

1. `{0: 3, 1: 2, 2: 2}` のようなmultilinear rankを指定すると、core shapeはなぜ `(3, 2, 2)` になるか
2. factor matrix \(U_n\) のshapeがなぜ `(元のmodeサイズ, rank)` になるか
3. `core + factors` の総要素数をどう計算するか
4. rankを下げるとparameter ratioとrelative errorがそれぞれどう変わったか
5. full HOSVDとpartial HOSVDでは、coreに残るdimensionがどう違うか
6. CNNのTucker-2で `out channel` / `in channel` のmodeだけを圧縮するのは、今回のpartial HOSVDとどう対応するか

ここまで説明できれば、次はCNNのConv2d重みにTucker-2を適用する段階へ進む。
